# LSTM, segunda versión

Mismos datos, mismas filas y mismo test de `6_modelo_lstm.ipynb`. Cambia lo que falló en la
primera versión, que repetía casi el mismo día toda la semana, con el mismo pico el sábado y el
domingo:

- La red es de codificador y decodificador. El codificador lee los últimos días y el
  decodificador arma la semana día por día, sabiendo qué día de la semana es cada uno y si es
  festivo.
- Cada paso de la entrada es un día con sus 24 horas, no una hora. Así puede mirar hasta 8
  semanas atrás sin volverse lenta.
- La pérdida es el error cuadrático, porque el criterio de la tesis es el RMSE; con MAE la red
  apunta a la mediana de cada hora y termina pronosticando un día típico.
- Los hiperparámetros se eligen con tres años de validación (2020, 2021 y 2022), como en
  XGBoost, y no con uno solo.
- El pronóstico es el promedio de 5 redes con semillas distintas.

Esta versión se diseñó después de ver la primera, incluida una semana del test, así que su
resultado en el test se reporta como exploratorio.

In [ ]:
%pip install -q keras-tuner

In [ ]:
import json
from pathlib import Path

import keras
import keras_tuner as kt
import numpy as np
import pandas as pd
from keras import layers
from scipy import stats

RUTA = Path("paneles/panel_lstm_modelo.csv.gz")
if not RUTA.exists():
    RUTA = Path("panel_lstm_modelo.csv.gz")
SALIDA = Path("resultados/modelo_lstm_v2")
SALIDA.mkdir(parents=True, exist_ok=True)
BUSQUEDA = Path("busqueda_lstm_v2")

HS = list(range(3, 10))
VARIABLES = ["cmg_hoy_vs_precio", "cmg_despacho", "disp_decl_hoy_rel", "brecha_escasez",
             "presion_hidro_horaria"]
HORARIAS = ["lp"] + VARIABLES
DIARIAS = ["dia_semana_sin", "dia_semana_cos", "es_festivo", "dia_ano_sin", "dia_ano_cos"]
VENTANAS = [14, 28, 56]
SEMILLAS = [0, 1, 2, 3, 4]

INICIO = "2015-03-01"
AÑOS_VALIDACION = [2020, 2021, 2022]
FIN_TEST = "2025-12-31"
FIN_2026 = "2026-10-06"
BLOQUES = {"h = 3 a 9": HS, "h = 3": [3], "h = 3 a 7": [3, 4, 5, 6, 7], "h = 8 a 9": [8, 9]}

CORRER_TEST = False
REENTRENAR = "mensual"

## Datos

Cada día es un vector con las 24 horas del precio y de las 5 variables, más su calendario. La
entrada del codificador son los últimos 14, 28 o 56 días; la del decodificador, el día de la
semana y el festivo de cada uno de los 7 días que se pronostican.

In [ ]:
panel = pd.read_csv(RUTA, parse_dates=["fecha"]).sort_values(["fecha", "hora"]).reset_index(drop=True)
faltan = [c for c in HORARIAS + DIARIAS + [f"y_h{h}" for h in HS] if c not in panel.columns]
assert not faltan, f"{RUTA} no es panel_lstm_modelo.csv.gz: le faltan {faltan[:5]}"
assert (panel.groupby("fecha").size() == 24).all()

f = pd.DatetimeIndex(panel["fecha"].drop_duplicates())
n_dias = len(f)
primera = panel["hora"] == 1
E_dia = np.concatenate([panel[HORARIAS].to_numpy(np.float32).reshape(n_dias, -1),
                        panel.loc[primera, DIARIAS].to_numpy(np.float32)], axis=1)
semana = panel.loc[primera, [f"dia_semana_k{k}" for k in range(1, 8)]].to_numpy(int)
festivo = panel.loc[primera, [f"es_festivo_k{k}" for k in range(1, 8)]].to_numpy(np.float32)
D_dia = np.concatenate([np.eye(7, dtype=np.float32)[semana], festivo[..., None]], axis=2)

Y_dia = (panel[[f"y_h{h}" for h in HS]].to_numpy(np.float32)
         .reshape(n_dias, 24, len(HS)).transpose(0, 2, 1))
LP = panel["lp"].to_numpy(np.float32).reshape(n_dias, 24)
Y_sem = np.full_like(Y_dia, np.nan)
for j, h in enumerate(HS):
    k = 7 * int(np.ceil(h / 7)) - h
    Y_sem[k:, j] = LP[:n_dias - k] - LP[k:]

falta = np.isnan(E_dia).any(axis=1).astype(int)


def origenes(desde, hasta, w, completos=True):
    con_historia = (np.arange(n_dias) >= w - 1) & (np.convolve(falta, np.ones(w, int))[:n_dias] == 0)
    m = con_historia & (f >= pd.Timestamp(desde))
    if completos:
        m &= (f + pd.Timedelta(days=max(HS)) <= pd.Timestamp(hasta)) & ~np.isnan(Y_dia).any(axis=(1, 2))
    else:
        m &= f <= pd.Timestamp(hasta)
    return np.flatnonzero(m)


def entradas(idx, w):
    X = np.stack([E_dia[i - w + 1:i + 1] for i in idx])
    return X, D_dia[idx], Y_dia[idx].reshape(len(idx), -1)


X, F, Y = entradas(origenes(INICIO, "2019-12-31", 28), 28)
print("codificador", X.shape, "decodificador", F.shape, "salida", Y.shape)

## Modelo

Codificador con una o dos capas LSTM. Su estado final arranca el decodificador, otra LSTM que
recorre los 7 días objetivo y saca las 24 horas de cada uno. Se busca el número de unidades,
de capas, el dropout, la tasa de Adam, el tamaño del lote y cuántos días mira hacia atrás.

In [ ]:
def escala(X, Y):
    mx, sx = X.reshape(-1, X.shape[-1]).mean(0), X.reshape(-1, X.shape[-1]).std(0) + 1e-6
    my, sy = Y.mean(0), Y.std(0) + 1e-6
    return (lambda X_: (X_ - mx) / sx), (lambda Y_: (Y_ - my) / sy), (lambda Z: Z * sy + my)


def construir(hp, w):
    unidades = hp.Int("unidades", 32, 256, step=32)
    drop = hp.Float("dropout", 0.1, 0.4, step=0.1)
    entrada = keras.Input(shape=(w, E_dia.shape[1]))
    futuro = keras.Input(shape=(len(HS), D_dia.shape[2]))
    x = entrada
    for _ in range(hp.Int("capas", 1, 2) - 1):
        x = layers.Dropout(drop)(layers.LSTM(unidades, return_sequences=True)(x))
    _, estado_h, estado_c = layers.LSTM(unidades, return_state=True)(x)
    y = layers.LSTM(unidades, return_sequences=True)(futuro, initial_state=[estado_h, estado_c])
    y = layers.Dense(24)(layers.Dropout(drop)(y))
    modelo = keras.Model([entrada, futuro], layers.Reshape((len(HS) * 24,))(y))
    tasa = hp.Float("tasa", 1e-4, 1e-2, sampling="log")
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=tasa), loss="mse")
    return modelo


def entrenar(hp, w, idx, epocas, idx_val=None, semilla=0):
    keras.utils.set_random_seed(semilla)
    X, F, Y = entradas(idx, w)
    nx, ny, volver = escala(X, Y)
    modelo = construir(hp, w)
    val, parar = None, []
    if idx_val is not None:
        Xv, Fv, Yv = entradas(idx_val, w)
        val = ([nx(Xv), Fv], ny(Yv))
        parar = [keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)]
    hist = modelo.fit([nx(X), F], ny(Y), validation_data=val, epochs=epocas,
                      batch_size=hp.get("lote"), callbacks=parar, verbose=0)
    return (modelo, nx, volver), hist


def pronosticar(redes, idx, w):
    X, F, _ = entradas(idx, w)
    p = [volver(modelo.predict([nx(X), F], verbose=0)) for modelo, nx, volver in redes]
    return np.mean(p, axis=0).reshape(len(idx), len(HS), 24)


def ensamble(hp, w, idx, epocas):
    return [entrenar(hp, w, idx, epocas, semilla=s)[0] for s in SEMILLAS]

## Búsqueda de hiperparámetros

Optimización bayesiana con 20 combinaciones. Cada una se entrena tres veces: hasta 2019 y se
mide en 2020, hasta 2020 y se mide en 2021, y hasta 2021 y se mide en 2022, con parada
temprana. Gana la de menor error promedio en los tres años. Las épocas que se usan después son
la mediana de las tres.

In [ ]:
BUSQUEDA.mkdir(exist_ok=True)
ruta_epocas = BUSQUEDA / "epocas.json"


class Busqueda(kt.BayesianOptimization):
    def run_trial(self, trial, *args, **kwargs):
        hp = trial.hyperparameters
        w = hp.Choice("dias", VENTANAS)
        hp.Choice("lote", [32, 64])
        perdidas, epocas = [], []
        for año in AÑOS_VALIDACION:
            tr = origenes(INICIO, f"{año - 1}-12-31", w)
            va = origenes(f"{año}-01-01", f"{año}-12-31", w)
            _, hist = entrenar(hp, w, tr, 100, va)
            perdidas.append(min(hist.history["val_loss"]))
            epocas.append(int(np.argmin(hist.history["val_loss"])) + 1)
        guardadas = json.loads(ruta_epocas.read_text()) if ruta_epocas.exists() else {}
        guardadas[trial.trial_id] = int(np.median(epocas))
        ruta_epocas.write_text(json.dumps(guardadas))
        return float(np.mean(perdidas))


buscador = Busqueda(max_trials=20, num_initial_points=8, seed=0, directory=str(BUSQUEDA),
                    project_name="lstm_v2")
buscador.search()

mejor_trial = buscador.oracle.get_best_trials(1)[0]
mejor = mejor_trial.hyperparameters
w = mejor.get("dias")
epocas = json.loads(ruta_epocas.read_text())[mejor_trial.trial_id]
pd.Series({**mejor.values, "epocas": epocas})

## Validación (2020-2022)

El ensamble de 5 redes se entrena en cada pliegue con la configuración elegida y se mide con el
criterio de la tesis. La fila de 2022 se compara con la validación de la primera versión.

In [ ]:
def media_y_ee(d, L):
    n = len(d)
    u = d - d.mean()
    v = np.mean(u * u)
    for l in range(1, min(L, n - 1) + 1):
        v += 2 * (1 - l / (L + 1)) * np.mean(u[l:] * u[:-l])
    return d.mean(), np.sqrt(v / n)


def criterio(idx, pred, tope):
    y = Y_dia[idx].copy()
    fuera = (f[idx].to_numpy()[:, None] + np.array(HS, dtype="timedelta64[D]")) > np.datetime64(tope)
    y[fuera] = np.nan
    filas = []
    for nombre, hs in BLOQUES.items():
        j = [HS.index(h) for h in hs]
        e = y[:, j] - pred[:, j]
        ingenuos = {"del día": y[:, j], "semanal": y[:, j] - Y_sem[idx][:, j]}
        rmse = {k: np.sqrt(np.nanmean(v ** 2)) for k, v in ingenuos.items()}
        ref = min(rmse, key=rmse.get)
        d = np.nanmean((e ** 2 - ingenuos[ref] ** 2).reshape(len(idx), -1), axis=1)
        m, ee = media_y_ee(d[~np.isnan(d)], max(hs) - 1 + 7)
        r = np.sqrt(np.nanmean(e ** 2))
        mejora, p = 100 * (1 - r / rmse[ref]), 2 * stats.norm.sf(abs(m / ee))
        filas.append(dict(bloque=nombre, rmse_log=r, mejor_ingenuo=ref, rmse_ingenuo=rmse[ref],
                          mejora=mejora, dm_t=m / ee, dm_p=p, funciona=mejora >= 5 and p < 0.05))
    return pd.DataFrame(filas).set_index("bloque").round(4)


partes = []
for año in AÑOS_VALIDACION:
    va = origenes(f"{año}-01-01", f"{año}-12-31", w)
    pred = pronosticar(ensamble(mejor, w, origenes(INICIO, f"{año - 1}-12-31", w), epocas), va, w)
    partes.append((va, pred))
    print(año)
    display(criterio(va, pred, f"{año}-12-31"))

idx = np.concatenate([p[0] for p in partes])
print("2020-2022")
criterio(idx, np.concatenate([p[1] for p in partes]), "2022-12-31")

## Test

Se corre una sola vez, con `CORRER_TEST = True` y la configuración ya cerrada. Se reentrena
cada mes, como en los otros modelos. Con 5 semillas son 230 entrenamientos; con
`REENTRENAR = "anual"` son 20.

In [ ]:
def walk_forward(desde, hasta):
    cortes = pd.date_range(desde, hasta, freq="MS" if REENTRENAR == "mensual" else "YS")
    tope = pd.Timestamp(hasta)
    partes = []
    for k, c in enumerate(cortes):
        sig = cortes[k + 1] if k + 1 < len(cortes) else tope + pd.Timedelta(days=1)
        tr = origenes(INICIO, c - pd.Timedelta(days=1), w)
        ev = origenes(c, min(sig - pd.Timedelta(days=1), tope - pd.Timedelta(days=min(HS))), w,
                      completos=False)
        partes.append((ev, pronosticar(ensamble(mejor, w, tr, epocas), ev, w)))
        print(c.date(), len(tr), "días de entrenamiento", flush=True)
    return np.concatenate([p[0] for p in partes]), np.concatenate([p[1] for p in partes])


def a_filas(idx, pred, tope):
    n = len(idx)
    d = pd.DataFrame({"fecha": np.repeat(f[idx], len(HS) * 24),
                      "hora": np.tile(np.arange(1, 25), n * len(HS)),
                      "h": np.tile(np.repeat(HS, 24), n),
                      "y_hat": pred.reshape(-1)})
    return d[d["fecha"] + pd.to_timedelta(d["h"], unit="D") <= pd.Timestamp(tope)]


if CORRER_TEST:
    resultados = []
    for nombre, desde, hasta in [("2023_2025", "2023-01-01", FIN_TEST), ("2026", "2026-01-01", FIN_2026)]:
        idx, pred = walk_forward(desde, hasta)
        a_filas(idx, pred, hasta).to_csv(SALIDA / f"prueba_{nombre}.csv.gz", index=False)
        resultados.append(criterio(idx, pred, hasta).assign(periodo=nombre))
    json.dump({**mejor.values, "epocas": epocas, "semillas": len(SEMILLAS), "perdida": "mse",
               "reentrenar": REENTRENAR}, open(SALIDA / "configuracion_final.json", "w"), indent=2)
    display(pd.concat(resultados))

## Semana final

Pronóstico del 9 al 15 de octubre de 2026 con el corte del 6 de octubre.

In [ ]:
if CORRER_TEST:
    corte = pd.Timestamp(FIN_2026)
    redes = ensamble(mejor, w, origenes(INICIO, corte, w), epocas)
    i = np.flatnonzero(f == corte)
    final = a_filas(i, pronosticar(redes, i, w), corte + pd.Timedelta(days=max(HS)))
    final["fecha_objetivo"] = final["fecha"] + pd.to_timedelta(final["h"], unit="D")
    final["precio_pron"] = panel.loc[(panel["fecha"] == corte), "precio"].to_numpy()[final["hora"] - 1] * np.exp(final["y_hat"])
    real = pd.read_csv("resultados/modelo_xgboost/precio_real_final.csv", parse_dates=["fecha_objetivo"])
    final = final.merge(real, on=["fecha_objetivo", "hora"])
    final.to_csv(SALIDA / "pronostico_final.csv", index=False)
    e = np.log(final["precio_real"]) - np.log(final["precio_pron"])
    print(f"RMSE log {np.sqrt(np.mean(e ** 2)):.3f}, MAPE "
          f"{100 * np.mean(np.abs(final['precio_real'] - final['precio_pron']) / final['precio_real']):.1f}%")